In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import json
import xgboost as xgb

from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

In [2]:
df = pd.read_csv('data/listings.csv.gz')

**Cleaning data**

In [3]:
to_drop_replaced = []
to_drop_low_info = []
to_drop_id = []
to_drop_leakage = []
checked_cols = []

Delete data with empty or almost empty columns  

In [4]:
missing = df.isnull().sum()
columns_empty = missing[missing > len(df)*0.9].index
df = df.drop(columns=columns_empty)

Delete data with missing price

In [5]:
df = df.drop(df[df.price.isnull()].index)

In [6]:
missing = df.isnull().sum()

Delete some unfinished offers. First to check if these are same raws

In [7]:
cols = missing[missing == 3].index

In [8]:
df = df.dropna(subset=cols)

Dropping the row with missing bathroom description

In [9]:
df = df.drop(df[df.bathrooms_text.isnull()].index)

Analyse what's left

In [10]:
missing = df.isnull().sum()

Cleaning price

In [11]:
df['price'] = df['price'].str.replace('$', '')
df['price'] = df['price'].str.replace(',', '')

df.price = pd.to_numeric(df.price)

**Feature engineering**

1. Host location (categories)

In [12]:
def host_location_category(loc): 
    if pd.isna(loc): 
        return 'unknown' 
    if 'Thessaloniki' in loc: 
        return 'thessaloniki' 
    if'Greece' in loc: 
        return 'greece'
    return 'abroad' 
    
df['host_location_cat'] = df['host_location'].apply(host_location_category) 

In [13]:
to_drop_replaced.append('host_location')

2. Reviews
   a) make a new column 'has_reviews'
   b) 'reviews_per_month' fill with 0

In [14]:
def has_reviews_category(loc): 
    if loc == 0: 
        return 0 
    return 1 
    
df['has_reviews'] = df['number_of_reviews'].apply(has_reviews_category) 

In [15]:
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

After splitting, fill all reviews columns with the median

In [16]:
checked_cols.append('has_reviews')
checked_cols.append('reviews_per_month')

In [17]:
missing = df.isnull().sum()

3. License

In [18]:
def license_ok(loc): 
    if pd.isna(loc): 
        return 0
    return 1 
    
df['license_ok'] = df['license'].apply(license_ok) 

4. Host description (make a flag if host wrote the description)

In [19]:
df['host_has_descr'] = (df['host_about'].str.len().fillna(0) >= 20).astype(int)
checked_cols.append('host_has_descr')
to_drop_replaced.append('host_about')

In [20]:
checked_cols.append('license_ok')
to_drop_replaced.append('license')

5. Amenities. (Create a new feature number of amenities)

In [21]:
df['n_amenities'] = df['amenities'].apply(json.loads).str.len()
to_drop_replaced.append('amenities')

6. Checking 'has availability'

Delete non unique columns

In [22]:
nunique = df.nunique()
columns_nonunique = nunique[nunique == 1].index
df = df.drop(columns=columns_nonunique)

7. First and last reviews. Cleaning dates (new columns - to count how many days from the first review and from the last, the days will be counted from the last scraped date)

In [23]:
df['days_since_last_review'] = (pd.to_datetime(df.last_scraped) - pd.to_datetime(df.last_review)).dt.days
df['days_since_last_review'].isnull().sum()

np.int64(501)

In [24]:
df['days_since_first_review'] = (pd.to_datetime(df.last_scraped) - pd.to_datetime(df.first_review)).dt.days
df['days_since_first_review'].isnull().sum()

np.int64(501)

In [25]:
to_drop_replaced.append('last_scraped')
to_drop_replaced.append('last_review')
to_drop_replaced.append('first_review')
checked_cols.append('days_since_first_review')
checked_cols.append('days_since_last_review')

8. Working with bathrooms categories. 87% of listings have exactly 1 bath, so creating only  2 new flags - Bathroom numbers (1 if more than 1, 0 if 1 or less) and Shared bathroom (=1 if shared)

In [26]:
df['bathrooms_num'] = df['bathrooms_text'].str.extract(r'(\d+\.?\d*)')[0].astype(float)
df['bathrooms_num'] = df['bathrooms_num'].fillna(0.5)
df['bathrooms_num'] = (df['bathrooms_num'] > 1).astype(int)

In [27]:
df['bathrooms_shared'] = df['bathrooms_text'].str.contains('shared', case=False).astype(int)
df['bathrooms_shared'].sum()
to_drop_replaced.append('bathrooms_text')

In [28]:
checked_cols.append('bathrooms_shared')
checked_cols.append('bathrooms_num')

**EDA**

Analyse price

Delete prices > 1000. 2 prices about 10000 euros are clearly placeholders; values above 1000 euros look unrealistic for Thessaloniki

In [29]:
df = df.drop(df[df.price > 1000].index)

Check if it looks like real listings (from 500 and up)

Apply log

In [30]:
df['log_price'] = np.log1p(df['price'])

Checking numerical columns (+ separate all 'id' columns)

In [31]:
numerical_cols = df.select_dtypes(include='number').columns

Remove from numerical columns all checked, all with id and all with prices

In [32]:
id_cols = [c for c in numerical_cols if c == 'id' or c.endswith('_id')]
numerical_cols = [c for c in numerical_cols if c not in id_cols  and c not in checked_cols and 'price' not in c]

In [33]:
def checking_num_col(s): 
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    outliners = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()
    max_share = (s.dropna() == s.max()).mean()
    skew = s.skew()
    null_quant = s.isnull().sum()
    negative_quant = (s < 0).sum()
    result = {'outliers': outliners, 'max_share': max_share, 'skew': skew, 'negative': negative_quant, 'isnull': null_quant}
    return result 

In [34]:
results = {col: checking_num_col(df[col]) for col in numerical_cols}
summary = pd.DataFrame(results).T

Checking skew

In [35]:
summary.sort_values('skew', ascending=False)

,outliers,max_share,skew,negative,isnull
minimum_nights,166.0,0.000217,31.121473,0.0,0.0
minimum_minimum_nights,166.0,0.000217,31.121473,0.0,0.0
minimum_nights_avg_ntm,311.0,0.000217,26.736613,0.0,0.0
maximum_minimum_nights,134.0,0.000217,24.991777,0.0,0.0
calculated_host_listings_count_shared_rooms,8.0,0.001733,23.968730,0.0,0.0
calculated_host_listings_count_private_rooms,496.0,0.004332,7.621078,0.0,0.0
host_listings_count,780.0,0.000217,5.179999,0.0,0.0
number_of_reviews,416.0,0.000217,4.247249,0.0,0.0
calculated_host_listings_count,711.0,0.043102,2.751235,0.0,0.0
calculated_host_listings_count_entire_homes,711.0,0.043102,2.747198,0.0,0.0


Delete 17 listings because they are long-term rentals

In [36]:
df = df[df['minimum_nights'] <= 30]

The column calculated_host_listings_count_shared_rooms to drop because only 8 non-zero values

In [37]:
to_drop_low_info.append('calculated_host_listings_count_shared_rooms')

Adding to the Drop list all columns with 'nights' except 'minimum_nights' because others have around the same skew and outliers

In [38]:
night_cols = [c for c in df.columns if 'nights' in c and c != 'minimum_nights']
to_drop_low_info.extend(night_cols)

We will aggregate 4 columns in 2, counting in month time as user and host and drop 4 old columns

In [39]:
df['host_tenure_month'] = df['hosts_time_as_host_months'] + df['hosts_time_as_host_years'] * 12
df['user_tenure_month'] = df['hosts_time_as_user_months'] + df['hosts_time_as_user_years'] * 12
host_user_columns = [c for c in df.columns if 'hosts_time_as_' in c]
to_drop_replaced.extend(host_user_columns)

'estimated_occupancy_l365d' derived from review counts with Inside Airbnb’s own assumptions; duplicates existing review features 

In [40]:
to_drop_low_info.append('estimated_occupancy_l365d')

Drop all columns we don't need

In [41]:
to_drop_id = id_cols
to_drop_leakage.append('estimated_revenue_l365d')
df =  df.drop(columns=(to_drop_id + to_drop_leakage + to_drop_replaced + to_drop_low_info))

Checking correlation with price

In [42]:
numerical_cols = df.select_dtypes(include='number').columns
numerical_cols = [c for c in numerical_cols if 'price' not in c]

No single numeric feature strongly predicts price (max correlation 0.496 with bedrooms); price depends on a combination of features 

Removing features that are highly correlated with each other

In [43]:
availability_cols = [c for c in df.columns if 'availability_' in c and c != 'availability_90']
reviews_cols = [c for c in df.columns if ('review_scores_' in c or 'number_of_reviews_l' in c) 
                and c != 'review_scores_rating' and c != 'review_scores_location']
to_drop_correlated = ['calculated_host_listings_count_entire_homes']
to_drop_correlated.extend(availability_cols)
to_drop_correlated.extend(reviews_cols)

In [44]:
df =  df.drop(columns=to_drop_correlated)

Drop all price_quote columns as well

In [45]:
to_drop_price_quote = [c for c in df.columns if c.startswith('price_quote')]
df =  df.drop(columns=to_drop_price_quote)

In [46]:
def select_categorical_cols(df):
    categorical_cols = df.select_dtypes(exclude='number').columns.tolist()
    return categorical_cols

In [47]:
to_drop_categorical_cols = [c for c in df.columns if c.endswith('_url')]
to_drop_categorical_cols.append('name')
to_drop_categorical_cols.append('host_name')
to_drop_categorical_cols.append('description')
to_drop_categorical_cols.append('calendar_last_scraped')
df = df.drop(columns=to_drop_categorical_cols)

In [48]:
to_drop_categorical_analysis = ['property_type'] 
df = df.drop(columns=to_drop_categorical_analysis) 

**Data split and missing values**

In [49]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [50]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.log_price.values
y_val = df_val.log_price.values
y_test = df_test.log_price.values

to_drop = ['price', 'log_price']
df_train = df_train.drop(columns=to_drop)
df_val = df_val.drop(columns=to_drop)
df_test = df_test.drop(columns=to_drop)

In [51]:
to_fill_median = df_train.columns[df_train.isnull().any()].tolist()

In [52]:
median_data = df_train[to_fill_median].median()
df_train = df_train.fillna(median_data)
df_val = df_val.fillna(median_data)
df_test = df_test.fillna(median_data)

***MODEL***

In [53]:
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')
test_dicts = df_test.to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)
features = dv.get_feature_names_out().tolist()
dtrain = xgb.DMatrix(X_train, label=y_train,feature_names=features) 
dval = xgb.DMatrix(X_val, label=y_val,feature_names=features) 
dtest = xgb.DMatrix(X_test, label=y_val,feature_names=features)

In [54]:
xgb_params = {
    'eta': 0.05,
    'max_depth': 6,
    'min_child_weight': 5,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}


xgb_final = xgb.train(xgb_params, dtrain, num_boost_round=301,
                      evals=[(dtrain, 'train'), (dval, 'val')],
                      verbose_eval=False)

xgb_rmse = root_mean_squared_error(y_val, xgb_final.predict(dval))
print('rmse val:', xgb_rmse)

rmse val: 0.29264047488672745


In [55]:
y_pred = xgb_final.predict(dtest)

rmse  = root_mean_squared_error(y_test, y_pred)